# Advanced Iterators and Generators

Deep dive into Python's **iterator protocol**, advanced generator patterns, **coroutines**, and `asyncio` basics.

### Topics
- Iterator protocol (`__iter__`, `__next__`)
- Custom iterator classes
- Generator pipelines and `yield from`
- Coroutines (`send`, `throw`, `close`)
- `asyncio` basics

> **Prerequisite**: Comprehensions and generators (Intermediate/03), functions.

## The Iterator Protocol

An **iterator** is any object that implements:
- `__iter__()` — returns the iterator object (often `self`)
- `__next__()` — returns the next value or raises `StopIteration`

Every Python `for` loop uses this protocol internally.

In [ ]:
# The iterator protocol in action
my_list = [1, 2, 3]

# What a for loop does internally
it = iter(my_list)  # calls __iter__()
print(f"Iterator object: {it}")
print(f"next(): {next(it)}")  # calls __next__()
print(f"next(): {next(it)}")
print(f"next(): {next(it)}")

try:
    next(it)  # raises StopIteration
except StopIteration:
    print("StopIteration raised — iterator exhausted")

# Using iter() with sentinel
def countdown(n):
    while n > 0:
        yield n
        n -= 1

gen = countdown(3)
print(f"\nList from generator: {list(gen)}")

## Custom Iterator Class

In [ ]:
# Custom iterator — RangeIterator
class RangeIterator:
    """Custom iterator mimicking range()."""
    def __init__(self, start, stop=None, step=1):
        if stop is None:
            start, stop = 0, start
        self.current = start
        self.stop = stop
        self.step = step
    
    def __iter__(self):
        return self  # iterators return themselves
    
    def __next__(self):
        if (self.step > 0 and self.current >= self.stop) or \
           (self.step < 0 and self.current <= self.stop):
            raise StopIteration
        value = self.current
        self.current += self.step
        return value

# Test
print("Custom RangeIterator:")
for i in RangeIterator(5):
    print(f"  {i}", end=" ")
print()

print("\nRangeIterator(2, 10, 2):")
for i in RangeIterator(2, 10, 2):
    print(f"  {i}", end=" ")
print()

## Generator Expressions vs List Comprehensions (Memory)

Generator expressions use **constant memory** while list comprehensions use **O(n)** memory.

In [ ]:
import sys

# Memory comparison
n = 1000000

# List comprehension — stores all in memory
lc = [x ** 2 for x in range(n)]
print(f"List comprehension size: {sys.getsizeof(lc) / 1024 / 1024:.1f} MB")

# Generator expression — lazy, tiny memory
ge = (x ** 2 for x in range(n))
print(f"Generator expression size: {sys.getsizeof(ge)} bytes")

# But iterating is similar in speed
import time

start = time.perf_counter()
total_lc = sum(x ** 2 for x in range(100000))  # generator in sum()
time_lc = time.perf_counter() - start

start = time.perf_counter()
total_ge = sum([x ** 2 for x in range(100000)])  # list first
time_ge = time.perf_counter() - start

print(f"\nGenerator in sum: {time_lc:.4f}s")
print(f"List then sum:    {time_ge:.4f}s")

## Generator Pipelines

In [ ]:
# Generator pipeline — processing streaming data
def lines_from_file(filename):
    """Generator: yield lines from a file."""
    with open(filename) as f:
        for line in f:
            yield line.strip()

def filter_comments(lines):
    """Generator: filter out comment lines."""
    for line in lines:
        if not line.startswith('#'):
            yield line

def parse_csv(lines, delimiter=','):
    """Generator: parse CSV lines into tuples."""
    for line in lines:
        if line:
            yield line.split(delimiter)

def filter_by_field(rows, field_idx, value):
    """Generator: filter rows by field value."""
    for row in rows:
        if row[field_idx] == value:
            yield row

def take(iterable, n):
    """Generator: yield first n items."""
    for i, item in enumerate(iterable):
        if i >= n:
            break
        yield item

# Create sample data
with open("pipeline_data.csv", "w") as f:
    f.write("# This is a comment\n")
    f.write("Alice,30,Engineer\n")
    f.write("Bob,25,Designer\n")
    f.write("# Another comment\n")
    f.write("Charlie,35,Engineer\n")
    f.write("Diana,28,Engineer\n")

# Build pipeline
pipeline = take(
    filter_by_field(
        parse_csv(
            filter_comments(
                lines_from_file("pipeline_data.csv")
            )
        ),
        2, "Engineer"
    ),
    2
)

print("Pipeline result (Engineers):")
for row in pipeline:
    print(f"  {row}")

# Memory: O(1) — processes one item at a time through the pipeline!
import os
os.remove("pipeline_data.csv")

## yield from in Depth

`yield from` delegates to a sub-iterator, simplifying flattening and chaining.

In [ ]:
# yield from — flattening with delegation
def flatten(seq):
    """Flatten arbitrarily nested sequences."""
    for item in seq:
        if hasattr(item, '__iter__') and not isinstance(item, (str, bytes)):
            yield from flatten(item)
        else:
            yield item

# Test
data = [1, [2, [3, 4], [5]], 6, [[7, 8], 9]]
print(f"Flattened: {list(flatten(data))}")

# yield from with generators — delegating
def chain_generators(*gens):
    """Chain multiple generators."""
    for gen in gens:
        yield from gen

g1 = (x for x in [1, 2, 3])
g2 = (x for x in [4, 5, 6])
g3 = (x for x in [7, 8, 9])

chained = list(chain_generators(g1, g2, g3))
print(f"Chained: {chained}")

## Coroutines: send(), throw(), close()

Generators can **receive values** via `send()`, making them **two-way** communication channels (coroutines).

In [ ]:
# Generator coroutine — using send()
def coroutine():
    """Coroutine that accumulates values sent to it."""
    total = 0
    count = 0
    while True:
        value = yield total / count if count > 0 else 0
        if value is not None:
            total += value
            count += 1

# Initialize coroutine (must send None first)
avg = coroutine()
next(avg)  # prime the coroutine (advance to first yield)

# Send values
print(f"After 10: {avg.send(10)}")
print(f"After 20: {avg.send(20)}")
print(f"After 30: {avg.send(30)}")

# throw() and close()
def controlled_coro():
    try:
        while True:
            x = yield
            print(f"Received: {x}")
    except GeneratorExit:
        print("Coroutine closing...")
    except ValueError:
        print("ValueError caught in coroutine!")

cc = controlled_coro()
next(cc)
cc.send("hello")
cc.throw(ValueError, "test error")  # throws exception inside generator
cc.close()  # raises GeneratorExit

## asyncio Basics

`asyncio` provides **asynchronous I/O** using `async`/`await` syntax. Key concepts:
- **Event loop**: Schedules and runs coroutines
- **Coroutine**: Defined with `async def`
- **Task**: Wraps a coroutine for concurrent execution
- **Future**: A result that will be available later

In [ ]:
# asyncio basics
import asyncio

# Define an async function (coroutine)
async def greet(name, delay):
    """Async function that simulates I/O with sleep."""
    await asyncio.sleep(delay)  # non-blocking sleep
    return f"Hello, {name}!"

# Run a single coroutine
result = asyncio.run(greet("Alice", 0.1))
print(f"Async result: {result}")

print("\n--- Multiple async tasks ---")

async def main():
    """Run multiple coroutines concurrently."""
    # Schedule two tasks to run concurrently
    task1 = asyncio.create_task(greet("Bob", 0.2))
    task2 = asyncio.create_task(greet("Charlie", 0.1))
    
    # Await both results
    result1 = await task1
    result2 = await task2
    
    print(f"Task 1: {result1}")
    print(f"Task 2: {result2}")
    
    # Or use asyncio.gather()
    results = await asyncio.gather(
        greet("David", 0.3),
        greet("Eve", 0.05),
    )
    print(f"Gathered: {results}")

asyncio.run(main())

## Interview Problems

### Problem 1: Custom Range Iterator
Implement an iterator that mimics `range()` with start, stop, and step.

In [ ]:
class Range:
    """Custom range class implementing the iterator protocol."""
    def __init__(self, *args):
        if len(args) == 1:
            self.start, self.stop, self.step = 0, args[0], 1
        elif len(args) == 2:
            self.start, self.stop, self.step = args[0], args[1], 1
        elif len(args) == 3:
            self.start, self.stop, self.step = args
        else:
            raise TypeError(f"Range takes 1-3 arguments ({len(args)} given)")
        if self.step == 0:
            raise ValueError("step must not be zero")
    
    def __iter__(self):
        self.current = self.start
        return self
    
    def __next__(self):
        if self.step > 0 and self.current >= self.stop:
            raise StopIteration
        if self.step < 0 and self.current <= self.stop:
            raise StopIteration
        value = self.current
        self.current += self.step
        return value
    
    def __len__(self):
        """Number of items in the range."""
        if self.step > 0 and self.start < self.stop:
            return max(0, (self.stop - self.start - 1) // self.step + 1)
        elif self.step < 0 and self.start > self.stop:
            return max(0, (self.start - self.stop - 1) // (-self.step) + 1)
        return 0

# Test
print("Custom Range tests:")
r1 = Range(5)
print(f"Range(5): {list(r1)}")
print(f"Range(5) len: {len(r1)}")

r2 = Range(2, 10, 2)
print(f"Range(2, 10, 2): {list(r2)}")
print(f"Range(2, 10, 2) len: {len(r2)}")

r3 = Range(10, 0, -3)
print(f"Range(10, 0, -3): {list(r3)}")

### Problem 2: Generator Pipeline for Data Processing
Build a pipeline that reads integers, filters, transforms, and aggregates.

In [ ]:
class DataPipeline:
    """A composable data processing pipeline using generators."""
    
    @staticmethod
    def from_iterable(data):
        for item in data:
            yield item
    
    @staticmethod
    def filter_even(source):
        for item in source:
            if item % 2 == 0:
                yield item
    
    @staticmethod
    def transform(source, func):
        for item in source:
            yield func(item)
    
    @staticmethod
    def take(source, n):
        for i, item in enumerate(source):
            if i >= n:
                break
            yield item
    
    @staticmethod
    def sum(source):
        total = 0
        for item in source:
            total += item
        return total

# Build pipeline
data = range(100)
pipeline = DataPipeline.transform(
    DataPipeline.take(
        DataPipeline.filter_even(
            DataPipeline.from_iterable(data)
        ),
        10
    ),
    lambda x: x ** 2
)

print("Pipeline: from 0-99, filter evens, take 10, square:")
print(f"  Result: {list(pipeline)}")

# Different pipeline: sum of first 5 odd squares
odd_squares = DataPipeline.transform(
    DataPipeline.take(
        (x for x in range(100) if x % 2 == 1),  # filter odd inline
        5
    ),
    lambda x: x ** 2
)
print(f"\nSum of first 5 odd squares: {DataPipeline.sum(odd_squares)}")

### Problem 3: Async Web Scraper (Simulated)
Simulate concurrent fetching of multiple URLs.

In [ ]:
import asyncio
import time

# Simulated async HTTP request
async def fetch_url(session_id, url):
    """Simulate fetching a URL asynchronously."""
    delay = hash(url) % 5 + 1  # deterministic "network delay" 1-5s
    print(f"  [Session {session_id}] Fetching {url}... (delay={delay}s)")
    await asyncio.sleep(delay)
    content = f"<html><body><h1>Content of {url}</h1></body></html>"
    print(f"  [Session {session_id}] Done fetching {url}")
    return {"url": url, "content": content, "status": 200}

async def scrape_urls(urls, max_concurrent=3):
    """Scrape multiple URLs concurrently with a concurrency limit."""
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def fetch_with_limit(session_id, url):
        async with semaphore:
            return await fetch_url(session_id, url)
    
    tasks = [
        fetch_with_limit(i, url)
        for i, url in enumerate(urls)
    ]
    
    results = await asyncio.gather(*tasks)
    return results

async def main():
    urls = [
        "https://example.com/page1",
        "https://example.com/page2",
        "https://example.com/page3",
        "https://example.com/page4",
        "https://example.com/page5",
    ]
    
    start = time.perf_counter()
    results = await scrape_urls(urls, max_concurrent=3)
    elapsed = time.perf_counter() - start
    
    print(f"\nFetched {len(results)} URLs in {elapsed:.2f}s")
    for r in results:
        print(f"  {r['url']} -> status {r['status']}")

asyncio.run(main())

# With 3 concurrent slots and 5 URLs (delays 1-5s), total time ~max(smallest 3 delays of 5)
# vs ~sum(delays) if sequential

## Summary

| Concept | Key Points |
|---------|------------|
| Iterator protocol | `__iter__()`, `__next__()`, `StopIteration` |
| Generator | Function with `yield` |
| `yield from` | Delegate to sub-iterator |
| Generator pipelines | Chain generators for streaming data |
| Coroutines | `send()`, `throw()`, `close()` |
| `async`/`await` | Asynchronous I/O with asyncio |

**Next**: Move to **03_Context_Managers.ipynb**.